# Chapter 15 - Understanding Strategy Risk

## Preparation

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
from scipy.stats import norm
from random import gauss
from itertools import product

from sklearn.datasets import make_classification
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC

from utils.sampling_bars import dollar_bar
from utils.backtest_stats import get_statistics, get_psr, get_dsr_s_star


%matplotlib inline
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = 16,6

## 1. A portfolio manager intends to launch a strategy that targets an annualized SR of 2. Bets have a precision rate of 60%, with weekly frequency. The exit conditions are 2% for profit-taking, and –2% for stop-loss.

### a. Is this strategy viable?

In [5]:
def annualized_symmetric_payouts(p, n):
    return (2 * p - 1) / (2 * np.sqrt(p * (1 - p))) * np.sqrt(n)

p = 0.6
n = 52

print(f"The annualized symmetric payout with precision {p} and frequency {n} is {annualized_symmetric_payouts(p, n)}")

The annualized symmetric payout with precision 0.6 and frequency 52 is 1.4719601443879742


Since the symmetric payout is 1.47, it is not viable to get an annualized SR of 2.

### b. Ceteris paribus, what is the required precision rate that would make the strategy profitable?

In [7]:
def symmetric_precision(sr, n):
    inner_term = n + sr ** 2
    a = 4 * inner_term
    b = -4 * inner_term
    c = n
    return (-b + np.sqrt(b ** 2 - 4 * a * c)) / (2 * a)

sr = 2
n = 52

print(f"The required precision rate that would make the strategy profitable for an annualized SR of {sr} and a frequency of {n} is {symmetric_precision(sr, n):4f}")

The required precision rate that would make the strategy profitable for an annualized SR of 2 and a frequency of 52 is 0.633631


### c. For what betting frequency is the target achievable?

In [10]:
def symmetric_frequency(sr, p):
    return int((4 * sr**2 * p * (1 - p)) / (2 * p - 1)**2)

sr = 2
p = 0.6

print(f"The required betting frequency that would make the strategy profitable for an annualized SR of {sr} and a precision rate of {p} is {symmetric_frequency(sr, p)}")

The required betting frequency that would make the strategy profitable for an annualized SR of 2 and a precision rate of 0.6 is 96


### d. For what profit-taking threshold is the target achievable?

In [13]:
def asymmetric_profit_taking(sr, sl, p, n):
    inner_term = sr * np.sqrt(p * (1 - p)) - np.sqrt(n) * p
    return (sl * (np.sqrt(n) + inner_term)) / inner_term

sr = 2
sl = -2
p = 0.6
n = 52

print(f"The required profit-taking threshold that would make the strategy profitable \nfor an annualized SR of {sr}, a stop-loss of {sl}, a precision rate of {p}, and a frequency of {n} is {asymmetric_profit_taking(sr, sl, p, n):4f}")

The required profit-taking threshold that would make the strategy profitable 
for an annualized SR of 2, a stop-loss of -2, a precision rate of 0.6, and a frequency of 52 is 2.309168


### e. What would be an alternative stop-loss?

In [14]:
def asymmetric_stop_loss(sr, pt, p, n):
    inner_term = sr * np.sqrt(p * (1 - p)) - np.sqrt(n) * p
    return (pt * inner_term) / (np.sqrt(n) + inner_term)

sr = 2
pt = 2
p = 0.6
n = 52

print(f"The required stop-loss threshold that would make the strategy profitable \nfor an annualized SR of {sr}, a profit-taking threshold of {pt}, a precision rate of {p}, and a frequency of {n} is {asymmetric_stop_loss(sr, pt, p, n):4f}")

The required stop-loss threshold that would make the strategy profitable 
for an annualized SR of 2, a profit-taking threshold of 2, a precision rate of 0.6, and a frequency of 52 is -1.732226


## 2. Following up on the strategy from exercise 1.

### a. What is the sensitivity of SR to a 1% change in each parameter?